# E33 --- de onde vem a cauda

O livro mediu a cauda dos mercados três vezes --- o recorde que não cai por pouco --- e achou três
assinaturas que não se parecem: razão pior/segundo de 1,277 no índice, 1,067 no Ibovespa, 1,956 no
Bitcoin. Este caderno pergunta de onde ela vem, com **duas famílias geradoras declaradas**:

- a que **sorteia**: dias independentes de cauda pesada (t de Student);
- a que **se reproduz**: cada dia ruim gerando dias ruins --- ramificação crítica em ambiente
  aleatório, imigrante repondo a extinção.

**Calibração declarada.** Cada mundo sai padronizado em desvio-padrão e é multiplicado pela escala
do mercado julgado; as assinaturas (razão entre extremos, número de recordes do lado da queda) não
dependem da escala.

**Assinaturas medidas.** (1) a razão pior/segundo em função do tamanho da amostra; (2) o número de
recordes contra o harmônico H(n) --- que para dias independentes é livre de distribuição. **Veredito
declarado:** a família reproduz o mercado quando o ponto do mercado (razão, recordes) cai dentro dos
percentis cinco a noventa e cinco da nuvem dela, no tamanho de amostra do próprio mercado.

**Simulação não vira resultado sobre o mundo** (AGENTS.md §8.5): os geradores medem qual das duas
histórias reproduce as assinaturas --- não qual delas é verdadeira.


In [1]:
# <- brinque com: MUNDOS, MUNDOS_CURVA, TAMANHOS, NUS, SEMENTE, SERIES
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import dados, graficos, ramificacao, recorde, volatilidade

RAIZ = Path.cwd()
SERIES = ("sp500.csv", "ibov.csv", "btc.csv")
ROTULOS_SERIES = {"sp500.csv": "sp", "ibov.csv": "ibov", "btc.csv": "btc"}
MUNDOS = 400          # mundos por família, na decisão por mercado
MUNDOS_CURVA = 200    # mundos por ponto da curva da razão
TAMANHOS = (250, 1000, 2500, 5000, 10000, 20000)
NUS = (3.0, 8.0)      # a cauda do t que sorteia: dois graus, para mostrar que a forma não é o expoente
SEMENTE = 71

SORTEIO = np.random.default_rng(SEMENTE)
print("frevolab %s | %d mundos por família | tamanhos %s" % (frevolab.VERSAO, MUNDOS, TAMANHOS))

frevolab 0.1.0 | 400 mundos por família | tamanhos (250, 1000, 2500, 5000, 10000, 20000)


## Painel 1 --- os mercados, medidos de novo

A razão entre o pior e o segundo pior dia de queda, o número de recordes do lado da queda, e o
harmônico que os dias independentes prometem --- nos três mercados, re-medidos aqui (o número que
abre o capítulo é do caderno do mundo que faltou, e a re-medida é parte da honestidade).

In [2]:
MERCADOS = {}
for arquivo in SERIES:
    preco = dados.carregar_serie(arquivo)
    retornos = volatilidade.retornos_log(preco).dropna()
    ext = ramificacao.extremos(retornos.to_numpy())
    MERCADOS[arquivo] = {
        "dias": int(retornos.size),
        "sigma": float(retornos.std()),
        "pior": ext["pior"], "segundo": ext["segundo"], "razao": ext["razao"],
        "recordes": recorde.conta(retornos.to_numpy(), lado="baixo"),
        "esperado": float(recorde.esperado(retornos.size)),
    }
    m = MERCADOS[arquivo]
    print("%-11s %5d dias | pior %.4f segundo %.4f razao %.3f | recordes %d contra H(n) %.2f"
          % (arquivo, m["dias"], m["pior"], m["segundo"], m["razao"], m["recordes"], m["esperado"]))

sp500.csv    6718 dias | pior 0.1277 segundo 0.0999 razao 1.277 | recordes 6 contra H(n) 9.39
ibov.csv     6620 dias | pior 0.1599 segundo 0.1499 razao 1.067 | recordes 8 contra H(n) 9.38
btc.csv      4387 dias | pior 0.4647 segundo 0.2376 razao 1.956 | recordes 6 contra H(n) 8.96


## Painel 2 --- a curva da razão, em função da amostra

A assinatura que separaria as histórias: se o pior dia é um sorteio solto, a razão entre os dois
piores não depende de quantos dias se olhou; se os dias grandes vêm de cacho, olhar mais dias acha
cachos maiores. Mede-se a mediana e a faixa de dez a noventa por cento, por tamanho de amostra,
nas duas famílias --- e nos dois graus do t, para mostrar que a forma não é o expoente.

In [3]:
CONFIGS = [("independente", {"nu": 3.0}), ("independente_oito", {"nu": 8.0}), ("reproducao", {})]
GERADORES = {"independente": ramificacao.independente, "independente_oito": ramificacao.independente,
             "reproducao": ramificacao.critica}
CURVA = {}
for rotulo, opcoes in CONFIGS:
    for n in TAMANHOS:
        razoes, quants = [], []
        for _ in range(MUNDOS_CURVA):
            mundo = GERADORES[rotulo](n, SORTEIO, **opcoes)
            razoes.append(ramificacao.extremos(mundo)["razao"])
            quants.append(recorde.conta(mundo, lado="baixo"))
        CURVA[(rotulo, n)] = {
            "razao_mediana": float(np.median(razoes)),
            "razao_piso": float(np.percentile(razoes, 10)),
            "razao_teto": float(np.percentile(razoes, 90)),
            "recordes_mediana": float(np.median(quants)),
        }
    print("%18s n=%6d | razao %.3f [%.3f, %.3f] | recordes %.2f contra H %.2f"
          % (rotulo, n, CURVA[(rotulo, n)]["razao_mediana"], CURVA[(rotulo, n)]["razao_piso"],
             CURVA[(rotulo, n)]["razao_teto"], CURVA[(rotulo, n)]["recordes_mediana"],
             recorde.esperado(n)))

      independente n= 20000 | razao 1.238 [1.037, 2.208] | recordes 10.00 contra H 10.48


 independente_oito n= 20000 | razao 1.121 [1.016, 1.393] | recordes 10.00 contra H 10.48


        reproducao n= 20000 | razao 1.173 [1.027, 1.507] | recordes 24.00 contra H 10.48


## Painel 3 --- a decisão: qual família reproduz o mercado

No tamanho de amostra do próprio mercado, quatrocentos mundos por família, na escala do mercado.
O ponto do mercado --- a razão e os recordes que ele tem --- contra a nuvem de cada família. O
veredito é o declarado no cabeçalho: dentro dos percentis cinco a noventa e cinco, nas duas
margens, a família reproduce o mercado.

In [4]:
NUVENS = {}
for arquivo, mercado in MERCADOS.items():
    n_m = mercado["dias"]
    for rotulo, opcoes in CONFIGS:
        razoes, quants = [], []
        for _ in range(MUNDOS):
            mundo = GERADORES[rotulo](n_m, SORTEIO, **opcoes) * mercado["sigma"]
            razoes.append(ramificacao.extremos(mundo)["razao"])
            quants.append(recorde.conta(mundo, lado="baixo"))
        NUVENS[(arquivo, rotulo)] = {
            "razao_mediana": float(np.median(razoes)),
            "razao_piso": float(np.percentile(razoes, 5)),
            "razao_teto": float(np.percentile(razoes, 95)),
            "recordes_mediana": float(np.median(quants)),
            "recordes_piso": float(np.percentile(quants, 5)),
            "recordes_teto": float(np.percentile(quants, 95)),
        }
    print("%-11s (%d dias) | mercado: razao %.3f, recordes %d" % (arquivo, n_m, mercado["razao"], mercado["recordes"]))
    for rotulo, _ in CONFIGS:
        nu = NUVENS[(arquivo, rotulo)]
        dentro = (nu["razao_piso"] <= mercado["razao"] <= nu["razao_teto"]
                  and nu["recordes_piso"] <= mercado["recordes"] <= nu["recordes_teto"])
        nu["dentro"] = bool(dentro)
        print("  %18s | razao %.3f [%.3f, %.3f] | recordes %.1f [%.1f, %.1f] | %s"
              % (rotulo, nu["razao_mediana"], nu["razao_piso"], nu["razao_teto"],
                 nu["recordes_mediana"], nu["recordes_piso"], nu["recordes_teto"],
                 "reproduz" if dentro else "não reproduz"))

sp500.csv   (6718 dias) | mercado: razao 1.277, recordes 6
        independente | razao 1.285 [1.021, 2.945] | recordes 9.0 [5.0, 14.0] | reproduz
   independente_oito | razao 1.108 [1.006, 1.567] | recordes 9.0 [5.0, 15.0] | reproduz
          reproducao | razao 1.178 [1.017, 1.761] | recordes 20.0 [15.0, 29.0] | não reproduz


ibov.csv    (6620 dias) | mercado: razao 1.067, recordes 8
        independente | razao 1.237 [1.015, 2.663] | recordes 9.0 [5.0, 14.0] | reproduz
   independente_oito | razao 1.119 [1.007, 1.565] | recordes 9.0 [6.0, 14.0] | reproduz
          reproducao | razao 1.148 [1.012, 1.616] | recordes 21.0 [15.0, 28.0] | não reproduz


btc.csv     (4387 dias) | mercado: razao 1.956, recordes 6
        independente | razao 1.249 [1.021, 2.571] | recordes 9.0 [5.0, 14.0] | reproduz
   independente_oito | razao 1.106 [1.007, 1.514] | recordes 9.0 [5.0, 13.0] | não reproduz
          reproducao | razao 1.168 [1.011, 1.753] | recordes 19.0 [13.0, 28.0] | não reproduz


## As figuras

In [5]:
# Figura 1: a razão e os recordes contra o tamanho da amostra, nas duas famílias.
CORES_FIG = {"independente": "#1f4e79", "independente_oito": "#7f7f7f", "reproducao": "#b03a2e"}
ROTULOS_FIG = {"independente": "t, três graus", "independente_oito": "t, oito graus",
               "reproducao": "ramificação crítica"}
eixo_n = np.array(TAMANHOS, dtype=float)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.4, 4.2))
for rotulo in CORES_FIG:
    med = [CURVA[(rotulo, n)]["razao_mediana"] for n in TAMANHOS]
    piso = [CURVA[(rotulo, n)]["razao_piso"] for n in TAMANHOS]
    teto = [CURVA[(rotulo, n)]["razao_teto"] for n in TAMANHOS]
    ax1.plot(eixo_n, med, color=CORES_FIG[rotulo], lw=1.6, label=ROTULOS_FIG[rotulo])
    ax1.fill_between(eixo_n, piso, teto, color=CORES_FIG[rotulo], alpha=0.15)
ax1.set_xscale("log")
ax1.set_xlabel("dias na amostra")
ax1.set_ylabel("razão pior/segundo (mediana e faixa)")
ax1.legend(frameon=False, fontsize=8)
ax2.plot(eixo_n, [recorde.esperado(n) for n in TAMANHOS], color="0.2", lw=1.2, ls="--",
         label="H(n), a promessa dos dias independentes")
for rotulo in CORES_FIG:
    ax2.plot(eixo_n, [CURVA[(rotulo, n)]["recordes_mediana"] for n in TAMANHOS],
             color=CORES_FIG[rotulo], lw=1.6, label=ROTULOS_FIG[rotulo])
ax2.set_xscale("log")
ax2.set_xlabel("dias na amostra")
ax2.set_ylabel("recordes de queda (mediana)")
ax2.legend(frameon=False, fontsize=8)
fig.tight_layout()
graficos.salvar(fig, "E33_cauda", 1)
plt.close(fig)

In [6]:
# Figura 2: a decisão, mercado por mercado --- a nuvem de cada família e o ponto do mercado.
fig, eixos = plt.subplots(1, 3, figsize=(9.4, 3.6), sharex=False)
for ax, arquivo in zip(eixos, SERIES):
    mercado = MERCADOS[arquivo]
    for rotulo in CORES_FIG:
        pontos = []
        for _ in range(120):
            mundo = GERADORES[rotulo](mercado["dias"], SORTEIO, **dict(CONFIGS[[r for r, _ in CONFIGS].index(rotulo)][1])) * mercado["sigma"]
            pontos.append((ramificacao.extremos(mundo)["razao"], recorde.conta(mundo, lado="baixo")))
        pontos = np.array(pontos)
        ax.scatter(pontos[:, 0], pontos[:, 1], s=6, color=CORES_FIG[rotulo], alpha=0.35,
                   label=ROTULOS_FIG[rotulo])
    ax.scatter([mercado["razao"]], [mercado["recordes"]], marker="*", s=180, color="black",
               zorder=5, label="o mercado")
    ax.set_title(ROTULOS_SERIES[arquivo], fontsize=10)
    ax.set_xlabel("razão pior/segundo")
    ax.set_ylabel("recordes de queda")
eixos[0].legend(frameon=False, fontsize=7, loc="upper right")
fig.tight_layout()
graficos.salvar(fig, "E33_cauda", 2)
plt.close(fig)

## Leitura visual das figuras

(Feita contra o PNG de cada figura, com o código ao lado.)

**Figura 1** --- à esquerda, as três curvas da razão são horizontais: a assinatura que separaria
as histórias não existe --- olhar mais dias não afasta o segundo pior do primeiro em nenhuma das
duas famílias (a azul e a cinza do t caem de pouco, a vermelha da ramificação fica parada). À
direita, a separação que a razão não fez, os recordes fazem: as duas curvas do t andam coladas no
harmônico tracejado (seis para dez, com o tracejado de seis para dez e meio), e a da ramificação
sai de junto dele e sobe sozinha para o dobro (dez para vinte e quatro, contra dez e meio do
tracejado) --- o cacho quebra recordes demais.

**Figura 2** --- nos três painéis a estrela do mercado (seis a oito recordes) cai na parte baixa
das nuvens do t e sempre abaixo das nuvens da ramificação, que moram acima dos treze recordes. No
painel do Bitcoin a estrela está à direita da nuvem cinza (o t de oito graus não faz razão de
duas), dentro do rabo azul do t de três --- a cauda tem de ser pesada de verdade para aquele
mercado caber. Os rabos direitos das nuvens azuis são longos (mundos em que o segundo pior veio
pequeno demais) --- e é essa largura toda que faz a razão, sozinha, ser evidência fraca.

In [7]:
# O resultado: um objeto por grandeza, para o livro citar por comando.
resultado = {
    "cauda_mundos": MUNDOS,
    "cauda_mundos_curva": MUNDOS_CURVA,
    "cauda_semente": SEMENTE,
    "cauda_recordes_esperado_inicio": round(float(recorde.esperado(TAMANHOS[0])), 2),
}
for arquivo, mercado in MERCADOS.items():
    rotulo = ROTULOS_SERIES[arquivo]
    for grandeza in ("dias", "pior", "segundo", "razao", "recordes", "esperado"):
        resultado["cauda_%s_%s" % (rotulo, grandeza)] = round(float(mercado[grandeza]), 4)
INICIO, FIM = TAMANHOS[0], TAMANHOS[-1]
NOME_EXTREMO = {INICIO: "inicio", FIM: "fim"}
for rotulo, _ in CONFIGS:
    for n in (INICIO, FIM):
        resultado["cauda_%s_razao_%s" % (rotulo, NOME_EXTREMO[n])] = round(CURVA[(rotulo, n)]["razao_mediana"], 3)
        resultado["cauda_%s_razao_%s_piso" % (rotulo, NOME_EXTREMO[n])] = round(CURVA[(rotulo, n)]["razao_piso"], 3)
        resultado["cauda_%s_razao_%s_teto" % (rotulo, NOME_EXTREMO[n])] = round(CURVA[(rotulo, n)]["razao_teto"], 3)
        resultado["cauda_%s_recordes_%s" % (rotulo, NOME_EXTREMO[n])] = round(CURVA[(rotulo, n)]["recordes_mediana"], 2)
    resultado["cauda_%s_recordes_esperado_fim" % rotulo] = round(float(recorde.esperado(FIM)), 2)
for (arquivo, rotulo), nu in NUVENS.items():
    serie_r = ROTULOS_SERIES[arquivo]
    for grandeza in ("razao_mediana", "razao_piso", "razao_teto", "recordes_mediana", "recordes_piso", "recordes_teto"):
        resultado["cauda_nuvem_%s_%s_%s" % (serie_r, rotulo, grandeza)] = round(float(nu[grandeza]), 3)
    resultado["cauda_nuvem_%s_%s_dentro" % (serie_r, rotulo)] = int(nu["dentro"])

# O que sai do laboratório e o que o livro cita: medida que o livro não usa é medida morta.
CITADAS_NO_LIVRO = ("cauda_btc_esperado",
    "cauda_btc_recordes",
    "cauda_ibov_esperado",
    "cauda_ibov_recordes",
    "cauda_independente_oito_razao_fim",
    "cauda_independente_oito_razao_inicio",
    "cauda_independente_razao_fim",
    "cauda_independente_razao_inicio",
    "cauda_independente_recordes_esperado_fim",
    "cauda_independente_recordes_fim",
    "cauda_independente_recordes_inicio",
    "cauda_mundos",
    "cauda_mundos_curva",
    "cauda_nuvem_btc_independente_oito_razao_teto",
    "cauda_nuvem_btc_independente_razao_mediana",
    "cauda_nuvem_btc_independente_razao_piso",
    "cauda_nuvem_btc_independente_razao_teto",
    "cauda_nuvem_btc_independente_recordes_mediana",
    "cauda_nuvem_btc_independente_recordes_piso",
    "cauda_nuvem_btc_independente_recordes_teto",
    "cauda_nuvem_btc_reproducao_razao_mediana",
    "cauda_nuvem_btc_reproducao_razao_piso",
    "cauda_nuvem_btc_reproducao_razao_teto",
    "cauda_nuvem_btc_reproducao_recordes_mediana",
    "cauda_nuvem_btc_reproducao_recordes_piso",
    "cauda_nuvem_btc_reproducao_recordes_teto",
    "cauda_nuvem_ibov_independente_razao_mediana",
    "cauda_nuvem_ibov_independente_razao_piso",
    "cauda_nuvem_ibov_independente_razao_teto",
    "cauda_nuvem_ibov_independente_recordes_mediana",
    "cauda_nuvem_ibov_independente_recordes_piso",
    "cauda_nuvem_ibov_independente_recordes_teto",
    "cauda_nuvem_ibov_reproducao_razao_mediana",
    "cauda_nuvem_ibov_reproducao_razao_piso",
    "cauda_nuvem_ibov_reproducao_razao_teto",
    "cauda_nuvem_ibov_reproducao_recordes_mediana",
    "cauda_nuvem_ibov_reproducao_recordes_piso",
    "cauda_nuvem_ibov_reproducao_recordes_teto",
    "cauda_nuvem_sp_independente_razao_mediana",
    "cauda_nuvem_sp_independente_razao_piso",
    "cauda_nuvem_sp_independente_razao_teto",
    "cauda_nuvem_sp_independente_recordes_mediana",
    "cauda_nuvem_sp_independente_recordes_piso",
    "cauda_nuvem_sp_independente_recordes_teto",
    "cauda_nuvem_sp_reproducao_razao_mediana",
    "cauda_nuvem_sp_reproducao_razao_piso",
    "cauda_nuvem_sp_reproducao_razao_teto",
    "cauda_nuvem_sp_reproducao_recordes_mediana",
    "cauda_nuvem_sp_reproducao_recordes_piso",
    "cauda_nuvem_sp_reproducao_recordes_teto",
    "cauda_recordes_esperado_inicio",
    "cauda_reproducao_razao_fim",
    "cauda_reproducao_razao_inicio",
    "cauda_reproducao_recordes_fim",
    "cauda_reproducao_recordes_inicio",
    "cauda_semente",
    "cauda_sp_esperado",
    "cauda_sp_recordes")
resultado = {chave: valor for chave, valor in resultado.items() if chave in CITADAS_NO_LIVRO}

caminho = Path("lab/resultados/E33_cauda.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))

lab/resultados/E33_cauda.json gravado | 58 grandezas
